In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

# 1. Device and Data Preparation
device = "cuda" if torch.cuda.is_available() else "cpu"
df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')

# Now we have 3 different variables (features)
features = ['AT_solar_generation_actual', 'AT_wind_onshore_generation_actual', 'AT_load_actual_entsoe_transparency']
data = df[features].dropna()

# We make NaN for erroneous readings below 4000 MW in consumption data.
data.loc[data['AT_load_actual_entsoe_transparency'] < 4000, 'AT_load_actual_entsoe_transparency'] = np.nan

# We fill in the gaps naturally according to the flow of time
data['AT_load_actual_entsoe_transparency'] = data['AT_load_actual_entsoe_transparency'].interpolate(method='time')

scaler = MinMaxScaler(feature_range=(-1, 1))
data_scaled = scaler.fit_transform(data.values)

def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback)])
        y.append(data[i + lookback])
    return np.array(X), np.array(y)

X, y = create_sequences(data_scaled, 24)
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float()), batch_size=64, shuffle=False)
X_test_device = torch.from_numpy(X_test).float().to(device)

# 2. Multivariate Model Unified in a Single Framework
class UnifiedMultiModel(nn.Module):
    def __init__(self, rnn_type):
        super(UnifiedMultiModel, self).__init__()
        bidir = (rnn_type == 'Bi-LSTM')
        rnn_class = nn.GRU if rnn_type == 'GRU' else nn.LSTM
        # input_dim=3, we strengthened the hidden layers

        self.rnn = rnn_class(3, 64, 2, batch_first=True, dropout=0.2, bidirectional=bidir)
        self.fc = nn.Linear(128 if bidir else 64, 3) # output_dim=3

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

gercek = scaler.inverse_transform(y_test)
predictions = {}
epochs = 10 # 3 değişken için ideal süre

print("--- Training Multivariate Models ---")
for name in ['LSTM', 'GRU', 'Bi-LSTM']:
    print(f"{name} eğitimi başladı...")
    model = UnifiedMultiModel(name).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    model.train()
    for epoch in range(epochs):
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            
    model.eval()
    with torch.no_grad():
        preds = model(X_test_device).cpu().numpy()
    # We return to Megawatt by fixing the negative values ​​to 0
    predictions[name] = np.clip(scaler.inverse_transform(preds), 0, None)

# 3. Comprehensive WAPE Metric Report
def print_metrics(y_t, y_p, feature_idx, feature_name):
    yt, yp = y_t[:, feature_idx], y_p[:, feature_idx]
    mae = mean_absolute_error(yt, yp)
    wape = np.sum(np.abs(yt - yp)) / np.sum(np.abs(yt)) * 100
    print(f"{feature_name} -> MAE: {mae:.2f} MW | WAPE: %{wape:.2f}")

print("\n--- MODEL COMPARISON REPORT ---")
for name in ['LSTM', 'GRU', 'Bi-LSTM']:
    print(f"\n[{name} SONUÇLARI]")
    print_metrics(gercek, predictions[name], 0, "Güneş  ")
    print_metrics(gercek, predictions[name], 1, "Rüzgar ")
    print_metrics(gercek, predictions[name], 2, "Tüketim")

# 4. Triple Visualization (Separate Graphics for Each)
titles = ['Güneş Enerjisi Üretimi', 'Rüzgar Enerjisi Üretimi', 'Elektrik Tüketimi (Şebeke Yükü)']
y_labels = ['Güneş (MW)', 'Rüzgar (MW)', 'Tüketim (MW)']

for i in range(3):
    plt.figure(figsize=(14, 5)) # Her döngüde yepyeni ve geniş bir grafik oluşturur
    
    # Real data and predictions of 3 models
    plt.plot(gercek[:100, i], label='Real', color='black', linewidth=3)
    plt.plot(predictions['LSTM'][:100, i], label='LSTM', linestyle='--', alpha=0.8)
    plt.plot(predictions['GRU'][:100, i], label='GRU', linestyle='-.', alpha=0.8)
    plt.plot(predictions['Bi-LSTM'][:100, i], label='Bi-LSTM', linestyle=':', linewidth=2, alpha=0.8)
    
    # Graphics settings
    plt.title(f'Avusturya {titles[i]} Tahmini (İlk 100 Saat)', fontsize=14, fontweight='bold')
    plt.xlabel('Time (Hour)', fontsize=12)
    plt.ylabel(y_labels[i], fontsize=12)
    plt.legend(loc='upper right', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    # Print each graphic separately on the screen
    plt.show()